In [ ]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import read_KPI_adequacy, apply_conservative_classification, load_solutions, combine_solutions

G_save = False

g_ORANGE = "Orange"
g_MODEL_VARIABLE = 'µ'
g_MODEL_VARIABLE_REFERENCE = 'mu_2'



In [ ]:
latex_textwidth_pt = 452.0 # single column
scale = 1
dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

def update_background(fig, legend_attr=legend_attr, dim=dim, showlegend=True):
    fig.update_layout(
        # plot_bgcolor="rgba(0,0,0,0)",
        template='simple_white',
        # yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        # xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        width=dim[0],
        height=dim[1],
        showlegend=showlegend,
        autosize=False,
        legend=legend_attr,  # Include legend attributes
        # font=dict(
        #     family="Computer Modern",  # or 'Arial', 'Courier New', etc.
        #     # size=12,                   # default font size for all text
        # #     # color="black"              # font color
        #     )
    )
    config = dict(showgrid=False, showticklabels=True, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey")
    # fig.for_each_yaxis(lambda yaxis: yaxis.update(**config))
    # fig.for_each_xaxis(lambda xaxis: xaxis.update(**config))
    fig.update_yaxes(**config)
    
    fig.update_xaxes(**config)
    
    fig.update_traces(
        boxmean=True,
        selector=dict(type='box')
    )
    n = 4
    fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
    # for axis in fig.layout:
    #     if axis.startswith('xaxis') or axis.startswith('yaxis'):
    #         fig.layout[axis].update(showticklabels=True, linecolor="grey", mirror=True)

    # for axis in fig.layout:
    #     if axis.startswith('xaxis') or axis.startswith('yaxis'):
    #         fig.layout[axis].update(title = '')


In [ ]:

ss = [
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v32.4s", 'model_type' : 'cumulative_reserve'},
    {'solution_folder': f"RTS-GMLC_v32.5s", 'model_type' : 'sensitivity'},
    # {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]

gcd_KPI_adequacy, gcdi_KPI_adequacy = read_KPI_adequacy(ss)
gcd_KPI_adequacy['E_ENS_%'] = gcd_KPI_adequacy['E_ENS_MWh']/gcd_KPI_adequacy['E_input_load_MWh']*100
gcd_KPI_adequacy['E_ENS_MWh'] = gcd_KPI_adequacy['E_ENS_MWh']*((gcd_KPI_adequacy['E_ENS_%'] >=0.01) + (gcd_KPI_adequacy['E_ENS_%'] <0.01))
gcd_KPI_adequacy['E_ENS_%']  = gcd_KPI_adequacy['E_ENS_%']*((gcd_KPI_adequacy['E_ENS_%'] >=0.01) + (gcd_KPI_adequacy['E_ENS_%'] <0.01))

# For cumulatibe reserve methodology only
if 'cumulative_reserve' in gcd_KPI_adequacy['model_type'].values:
    gcd_KPI_adequacy.loc[gcd_KPI_adequacy['model_type'] == 'cumulative_reserve', 'µ'] = 'mu_3'
if 'e-reserve' in gcd_KPI_adequacy['model_type'].values:
    gcd_KPI_adequacy.loc[gcd_KPI_adequacy['model_type'] == 'e-reserve', 'µ'] = 'mu_4'


gcd_KPI_adequacy['µ_numeric'] = gcd_KPI_adequacy['µ'].str.replace('mu_', '').str.replace('_', '.').astype(float)


Data construction and filtering

In [ ]:
missing_days = set(gcd_KPI_adequacy[gcd_KPI_adequacy.model_type == 'envelope']['day']) - set(gcd_KPI_adequacy[gcd_KPI_adequacy.model_type == 'sensitivity']['day'])
print(missing_days)


In [ ]:
print(len(missing_days))

In [ ]:
filter_ = gcd_KPI_adequacy.pivot(
    index=['day'],
    columns = [g_MODEL_VARIABLE],
    values=['solution_id']
).dropna().index
gcd_KPI_adequacy = gcd_KPI_adequacy[gcd_KPI_adequacy['day'].isin(filter_)]
gcdi_KPI_adequacy = gcdi_KPI_adequacy[gcdi_KPI_adequacy['day'].isin(filter_)]
print(gcd_KPI_adequacy['day'].nunique())

# filter_ = (gcd_KPI_adequacy.model_type == 'envelope')&(gcd_KPI_adequacy.E_ENS_MWh <=0.1)&(gcd_KPI_adequacy.E_LGEN_MWh <= 0.1)
# filter_ = (gcd_KPI_adequacy.model_type == 'envelope')&((gcd_KPI_adequacy.E_ENS_MWh >0.1)|(gcd_KPI_adequacy.E_LGEN_MWh > 0.1))
# days= gcd_KPI_adequacy.loc[filter_].day.unique()
# gcd_KPI_adequacy = gcd_KPI_adequacy[gcd_KPI_adequacy['day'].isin(days)]
# print(days.size)

In [ ]:
# Select best mu per day for cumulative_reserve vs envelope using gcd_KPI_adequacy
# For MPhil this extra filter (gcd_KPI_adequacy["µ"] == "mu_2") was not implemented. However, if I add it, there are no changes on results.
base = (gcd_KPI_adequacy[(gcd_KPI_adequacy["model_type"] == "envelope") ][ #(gcd_KPI_adequacy["µ"] == "mu_2")
    ["day", "OV_uc", "E_ENS_MWh"]
].rename(columns={"OV_uc": "OV_uc_env", "E_ENS_MWh": "E_ENS_MWh_env"}))

cand = gcd_KPI_adequacy[gcd_KPI_adequacy["model_type"] == "sensitivity"].merge(
    base, on="day", how="inner"
)

def pick_best(group: pd.DataFrame) -> pd.Series:
    feasible = group[group["E_ENS_MWh"] <= group["E_ENS_MWh_env"]]
    if not feasible.empty:
        best = feasible.sort_values(["OV_uc", "E_ENS_MWh"]).iloc[0].copy()
        best['selection_category'] = 'feasible'
    else:
        # If no feasible solutions, pick lowest cost where OV_uc >= OV_uc_env
        infeasible = group[group["OV_uc"] >= group["OV_uc_env"]]
        if not infeasible.empty:
            best = infeasible.sort_values(["OV_uc", "E_ENS_MWh"]).iloc[0].copy()
            best['selection_category'] = 'infeasible_with_cost_constraint'
        else:
            # Fallback: pick lowest cost from all solutions
            # Solution will never enter here as fully conservative ensures: OV_uc(mu=1) >= OV_uc_env
            best = group.sort_values(["OV_uc", "E_ENS_MWh"]).iloc[0].copy()
            best['selection_category'] = 'fallback'
    return best

best_u_by_day = cand.groupby("day", group_keys=False).apply(pick_best).reset_index(drop=True)

# Print statistics about selection categories
print("\n=== Selection Statistics ===")
print(f"Total number of days: {len(best_u_by_day)}")
print("\nBreakdown by selection category:")
category_counts = best_u_by_day['selection_category'].value_counts()
for category, count in category_counts.items():
    percentage = (count / len(best_u_by_day)) * 100
    print(f"  {category}: {count} ({percentage:.1f}%)")
print("="*30 + "\n")
best_u_by_day = best_u_by_day.assign(
    model_type="sensitivity",
    **{g_MODEL_VARIABLE: "mu_5"}
)

# Append into gcd_KPI_adequacy (excluding selection_category column)
gcd_KPI_adequacy = pd.concat(
    [gcd_KPI_adequacy, best_u_by_day[gcd_KPI_adequacy.columns]],
    ignore_index=True
)





In [ ]:
show = best_u_by_day[["day", g_MODEL_VARIABLE, "OV_uc", "E_ENS_MWh", "OV_uc_env", "E_ENS_MWh_env"]]
show[show["E_ENS_MWh"] > show["E_ENS_MWh_env"]]

In [ ]:
cost_columns = ['OV_uc', 'thermal_production_cost_uc', 'E_OPEX', 'E_ENS_MWh', 'E_LLD_h']
# cost_differences = pd.DataFrame(index=filter_)
differences = gcd_KPI_adequacy.pivot(
    index=['day'],
    columns = [g_MODEL_VARIABLE],
    values=cost_columns + ['E_input_load_MWh'],
    # var_name='cost_type',
    # value_name='cost_value'
)
for col in cost_columns:
        for mu in gcd_KPI_adequacy[gcd_KPI_adequacy[g_MODEL_VARIABLE] != g_MODEL_VARIABLE_REFERENCE][g_MODEL_VARIABLE].unique():
            differences[(col+'_diff_%',mu)] = (differences[(col,mu)] - differences[(col,g_MODEL_VARIABLE_REFERENCE)]) / differences[(col,g_MODEL_VARIABLE_REFERENCE)]*100
            differences[(col+'_diff',mu)] = (differences[(col,mu)] - differences[(col,g_MODEL_VARIABLE_REFERENCE)])


differences = differences.stack(future_stack=True).dropna(how = 'all').reset_index()
differences['E_ENS_MWh_diff_%'] = differences['E_ENS_MWh_diff']/differences['E_input_load_MWh']*100
differences['E_LLD_h_diff_%'] = differences['E_LLD_h_diff']/24*100

def mu_sort_key(mu_label):
    if isinstance(mu_label, str) and mu_label.startswith('mu_'):
        num_str = mu_label[3:].replace('_', '.')
        try:
            return float(num_str)
        except ValueError:
            return float('inf')
    return float('inf')

mu_order = sorted(differences[g_MODEL_VARIABLE].dropna().unique(), key=mu_sort_key)
differences[g_MODEL_VARIABLE] = pd.Categorical(
    differences[g_MODEL_VARIABLE], categories=mu_order, ordered=True
 )
differences = differences.sort_values([g_MODEL_VARIABLE, 'day']).reset_index(drop=True)
# Filter to keep only rows where envelope (mu_2) has lower or equal unserved energy
# differences = differences[differences['E_ENS_MWh_diff_%']<=0]


In [ ]:
legend_attr = dict(
    x=0.5,
    y=-0.4,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

to_plot_name_map = {'OV_uc_diff_%': 'DA Cost Difference [%/day]',
                    'E_ENS_MWh_diff_%': 'RT Unserved Energy [%/day]',
                    }


to_plot = differences.melt(
    id_vars=['day', g_MODEL_VARIABLE],
    value_vars=[    
                'OV_uc_diff_%',
                # 'E_ENS_MWh_diff_%',
                'E_ENS_MWh_diff_%',
                # 'E_LLD_h_diff_%'
                # 'E_LLD_h_diff'
                ],
    )

to_plot = to_plot.merge(gcd_KPI_adequacy[[g_MODEL_VARIABLE, 'day', 'µ_numeric']], on = ['day', g_MODEL_VARIABLE], how = 'left')
to_plot['variable'] = to_plot['variable'].replace(to_plot_name_map)
# facet_mapping = {k:v for (k,v) in to_plot_name_map.items() if k in to_plot['variable'].unique()}

fig = px.box(
    to_plot[to_plot['µ_numeric']<1],
    # x='µ_numeric',
    y='value',
    color='µ_numeric',
    # range_color=(to_plot["µ_numeric"].min(), to_plot["µ_numeric"].max()),
    facet_col='variable',
    
    # category_orders={
    #     "model_type": ["conservative", "dynamic", "e-reserve", "stochastic"],
    #     g_MODEL_VARIABLE: mu_order,
    #     "variable": list(facet_mapping.values())},
    labels = {'µ_numeric': 'multiplier (µ)'},
    # points="all",  # show all points
    # facet_order = list(to_plot_name_map.values()),
    boxmode="group",
    facet_col_spacing=0.07,
    # hover_dat
)

dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*.6)

legend_attr = dict(
    x=0.5,
    y=-0.3,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)


update_background(fig,  legend_attr, dim)
fig.update_yaxes(matches=None)
# fig.layout['yaxis2'].update(showticklabels=False)

for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(title = '')

for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        # annotation.text = annotation.text.replace(" ($/day)", "<br>[$/day]").replace(" [$/day]", "<br>[$/day]").replace(" [%/day]", "<br>[%/day]").replace(" [h/day]", "<br>[h/day]")
        annotation.text = annotation.text.replace("variable=", "")
        # annotation.text = ""

# n = 4
# fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()
# if G_save:
#     fig.write_image("day_ahead_cost_&_unserved_energy_per_mu.pdf", width=dim[0], height=dim[1])

In [ ]:
legend_attr = dict(
    x=0.5,
    y=-0.4,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

to_plot_name_map = {'OV_uc_diff_%': 'DA Cost Difference [%/day]',
                    'E_ENS_MWh_diff_%': 'RT Unserved Energy [%/day]',
                    }

to_plot = differences.melt(
    id_vars=['day', g_MODEL_VARIABLE],
    value_vars=[
                'OV_uc_diff_%',
                'E_ENS_MWh_diff_%',
                ],
)

to_plot = to_plot.merge(gcd_KPI_adequacy[[g_MODEL_VARIABLE, 'day', 'µ_numeric']],
                        on=['day', g_MODEL_VARIABLE], how='left')
to_plot['variable'] = to_plot['variable'].replace(to_plot_name_map)

facet_labels = [
    to_plot_name_map[key]
    for key in to_plot_name_map
    if to_plot_name_map[key] in to_plot['variable'].unique()
]

manual_zoom_ranges = {
    "DA Cost Difference [%/day]": (-2.5, 6.0),
    "RT Unserved Energy [%/day]": (-0.5, 4),
}
zoom_ranges = {}
for label, group in to_plot.groupby("variable"):
    if label in manual_zoom_ranges:
        zoom_ranges[label] = manual_zoom_ranges[label]
        continue

    # fallback if you forgot a label
    values = group["value"].dropna()
    if values.empty:
        continue
    low = values.quantile(0.02)
    high = values.quantile(0.8)
    if low == high:
        pad = max(abs(low) * 0.1, 1e-6)
        low -= pad
        high += pad
    zoom_ranges[label] = (low, high)

fig_full = px.box(
    to_plot[to_plot['µ_numeric'] < 1],
    x='µ_numeric',
    y='value',
    color='µ_numeric',
    facet_col='variable',
    labels={'µ_numeric': 'multiplier (µ)'},
    boxmode="group",
    facet_col_spacing=0.07,
    category_orders={"variable": facet_labels},
)

fig_zoom = px.box(
    to_plot[to_plot['µ_numeric'] < 1],
    x='µ_numeric',
    y='value',
    color='µ_numeric',
    facet_col='variable',
    labels={'µ_numeric': 'multiplier (µ)'},
    boxmode="group",
    facet_col_spacing=0.07,
    category_orders={"variable": facet_labels},
)

ncols = max(len(facet_labels), 1)
subplot_titles = facet_labels + ["" for _ in facet_labels]
fig_combined = make_subplots(
    rows=2,
    cols=ncols,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.07,
    vertical_spacing=0.05,
)

for trace in fig_full.data:
    col = 2 if getattr(trace, "xaxis", "x") == "x2" else 1
    fig_combined.add_trace(trace, row=1, col=col)

for trace in fig_zoom.data:
    trace.update(showlegend=False)
    col = 2 if getattr(trace, "xaxis", "x") == "x2" else 1
    fig_combined.add_trace(trace, row=2, col=col)

for col, label in enumerate(facet_labels, start=1):
    if label in zoom_ranges:
        fig_combined.update_yaxes(range=zoom_ranges[label], row=2, col=col)

dim = (latex_textwidth_pt * scale, latex_textwidth_pt * scale*1)

# legend_attr = dict(
#     x=0.5,
#     y=-0.3,
#     yanchor="bottom",
#     xanchor="center",
#     orientation="v"
# )



# fig_combined.update_yaxes(matches=True)

update_background(fig_combined, legend_attr, dim)
fig_combined.update_layout(showlegend=False,)
# for axis in fig_combined.layout:
#     if axis.startswith('x2axis') or axis.startswith('yaxis'):
#         fig_combined.layout[axis].update(title='')

for col in range(1, ncols + 1):
    fig_combined.update_xaxes(showticklabels=False, row=1, col=col)
    # fig_combined.update_xaxes(showticklabels=True, row=1, col=2)
    fig_combined.update_xaxes(title_text="multiplier (µ)", row=2, col=col)
fig_combined.update_annotations(font=dict(size=12))
fig_combined.show()
if G_save:
    fig_combined.write_image("day_ahead_cost_&_unserved_energy_per_mu.pdf", width=dim[0], height=dim[1])

In [ ]:
yearly_cost = gcd_KPI_adequacy.groupby('model_type')[['start_cost_uc', 'fixed_cost_uc', 'production_cost_uc', 'storage_production_cost_uc', 'LGEN_cost_uc', 'LOL_cost_uc']].sum()*(1e-6)
yearly_cost.round(2)

In [ ]:
# ecdf = compute_ecdf(to_plot, value_col="value", group_cols=[g_MODEL_VARIABLE, "variable"])
# Create the ecdf plot
fig = px.ecdf(
    to_plot,
    x='value',
    color=g_MODEL_VARIABLE,
    hover_data='day',
    facet_col='variable',
    facet_col_spacing=0.07,
    labels = {'value': '', g_MODEL_VARIABLE: 'model type'},
)

# update_background(fig, legend_attr, dim)
fig.update_xaxes(matches=None)
# update_background(fig, legend_attr, dim)
for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(title = '')

fig.update_layout(yaxis1=dict(title ='Cumulative probability')),
# fig.update_layout(xaxis1=dict(title ='%/day')),
# fig.update_layout(xaxis2=dict(title ='%/day')),

for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")
        # annotation.text = ""
# n = 4
# fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()



In [ ]:
# if G_save:
#     fig.write_image("day_ahead_cost_&_unserved_energy.pdf", width=dim[0], height=dim[1])

In [ ]:
# left = gcd_KPI_adequacy.melt(
#     id_vars=['day', g_MODEL_VARIABLE],
#     value_vars=['µ_numeric'])
# aux = pd.concat([left, to_plot], axis=0)


In [ ]:

fig = px.ecdf(
    to_plot[to_plot['µ'] == 'mu_5'].replace({'µ': {'mu_5': 'µ*'}}),
    x='value',
    color='µ',
    hover_data='day',
    facet_col='variable',
    facet_col_spacing=0.09,
    labels = {'µ': 'envelope'},
    color_discrete_sequence =['green'],
)

legend_attr = dict(
    x=0.5,
    y=-0.3,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)
dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*.6)
update_background(fig, legend_attr, dim)


for axis in fig.layout:
    if axis.startswith('xaxis') or axis.startswith('yaxis'):
        fig.layout[axis].update(title = '')

fig.update_layout(yaxis1=dict(title ='Cumulative probability'))
fig.update_xaxes(matches=None)
# fig.update_xaxes(range=[-50, 30], row=1, col=1)
# fig.update_xaxes(range=[-4, 2], row=1, col=2)
fig.update_xaxes(dtick=15, row=1, col=1)
fig.update_xaxes(dtick=1, row=1, col=2)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")
n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()
if G_save:
    fig.write_image("day_ahead_cost_&_unserved_energy_mu_best.pdf", width=dim[0], height=dim[1])

In [ ]:

# Filter for mu_5 cases only
mu5_differences = differences[differences[g_MODEL_VARIABLE] == 'mu_5'].copy()

# Filter for cases with improvements (negative differences)
# mu5_improvements = mu5_differences[
#     (mu5_differences['OV_uc_diff_%'] < 0) | (mu5_differences['E_ENS_MWh_diff_%'] < 0)
# ].copy()
mu5_improvements = mu5_differences.copy()

# Merge with gcd_KPI_adequacy to get µ_numeric
mu5_improvements = mu5_improvements.merge(
    gcd_KPI_adequacy[['day', g_MODEL_VARIABLE, 'µ_numeric']].drop_duplicates(),
    on=['day', g_MODEL_VARIABLE],
    how='left'
)

# Summary statistics
print("Summary statistics of µ_numeric for mu_5 improvements:")
print(mu5_improvements['µ_numeric'].describe())

# Categorize by improvement type
mu5_improvements['improvement_type'] = np.select(
    [(mu5_improvements['OV_uc_diff_%'] < 0) & (mu5_improvements['E_ENS_MWh_diff_%'] < 0),
     (mu5_improvements['OV_uc_diff_%'] < 0) & (mu5_improvements['E_ENS_MWh_diff_%'] >= 0),
     (mu5_improvements['OV_uc_diff_%'] >= 0) & (mu5_improvements['E_ENS_MWh_diff_%'] < 0)],
    ['Cost & Reliability', 'Cost only', 'Reliability only'],
    default='No improvement'
)

In [ ]:


fig2 = px.box(
    mu5_improvements,
    x='µ_numeric',
    color='improvement_type',
    labels={'µ_numeric': 'Distribution of µ* by improvement type', 'improvement_type': 'Improvement'},
    # title='Distribution of µ_best by improvement type',
)

dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)
legend_attr = dict(
    x=0.5,
    y=1.05,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

update_background(fig2, legend_attr, dim)
n = 4
fig2.update_layout(margin=dict(l=n+30, r=n+30, t=n+30, b=n+30))
fig2.show()
if G_save:
    fig2.write_image("mu_best_distribution_by_improvement.pdf", width=dim[0], height=dim[1])

In [ ]:

px.scatter(differences, x = 'OV_uc_diff_%', y = 'E_ENS_MWh_diff_%', color = g_MODEL_VARIABLE, hover_data = 'day')

In [ ]:
# Quadrant distribution for OV_uc_diff_% vs E_ENS_MWh_diff_% with symmetric tolerance band
epsilon = .1  # percentage points
quadrant_df = differences[['day', g_MODEL_VARIABLE, 'OV_uc_diff_%', 'E_ENS_MWh_diff_%']].dropna().copy()
x = quadrant_df['OV_uc_diff_%']
y = quadrant_df['E_ENS_MWh_diff_%']

x_band = x.abs() <= epsilon
y_band = y.abs() <= epsilon
x_pos = x > epsilon
x_neg = x < -epsilon
y_pos = y > epsilon
y_neg = y < -epsilon

quadrant_df['quadrant'] = np.select(
    [x_pos & y_pos,
     x_neg & y_pos,
     x_neg & y_neg,
     x_pos & y_neg,
     x_band & y_band,
     x_band & y_pos,
     x_band & y_neg,
     x_pos & y_band,
     x_neg & y_band],
    [f"x> {epsilon}, y> {epsilon}",
     f"x< -{epsilon}, y> {epsilon}",
     f"x< -{epsilon}, y< -{epsilon}",
     f"x> {epsilon}, y< -{epsilon}",
     f"|x|<= {epsilon}, |y|<= {epsilon}",
     f"|x|<= {epsilon}, y> {epsilon}",
     f"|x|<= {epsilon}, y< -{epsilon}",
     f"x> {epsilon}, |y|<= {epsilon}",
     f"x< -{epsilon}, |y|<= {epsilon}"],
    default=f"|x|<= {epsilon}, |y|<= {epsilon}"
 )

quadrant_counts = (quadrant_df.groupby([g_MODEL_VARIABLE, 'quadrant'])
                   .size()
                   .rename('count')
                   .reset_index())

quadrant_totals = quadrant_counts.groupby(g_MODEL_VARIABLE)['count'].transform('sum')
quadrant_counts['pct'] = quadrant_counts['count'] / quadrant_totals * 100

order = [
    f"x> {epsilon}, y> {epsilon}",
    f"x< -{epsilon}, y> {epsilon}",
    f"x< -{epsilon}, y< -{epsilon}",
    f"x> {epsilon}, y< -{epsilon}",
    f"|x|<= {epsilon}, |y|<= {epsilon}",
    f"|x|<= {epsilon}, y> {epsilon}",
    f"|x|<= {epsilon}, y< -{epsilon}",
    f"x> {epsilon}, |y|<= {epsilon}",
    f"x< -{epsilon}, |y|<= {epsilon}",
]

fig_quadrants = px.bar(
    quadrant_counts,
    x=g_MODEL_VARIABLE,
    y='pct',
    color='quadrant',
    category_orders={g_MODEL_VARIABLE: mu_order, 'quadrant': order},
    labels={'pct': 'Share of points (%)', g_MODEL_VARIABLE: 'model type'},
    text=quadrant_counts['pct'].round(1),
)
fig_quadrants.update_layout(barmode='stack')
fig_quadrants.update_traces(textposition='inside')
fig_quadrants.show()


Positive cases are
- |x|<=0.1&y<-0.1 : no cost impact and less unserved energy
- x<-0.1&|y|<=0.1 : cost reduction and no impact on unserved energy

Negative cases:
- x<-0.1&y>0.1 : cost reduction but more unserved energy (underestimation of multiplier)
- |x|<=0.1&y>0.1 : no cost impact but more unserved energy (very bad)
- x>0.1&y<-0.1 :  cost increse with less unserved energy (overestimation of multiplier)
- x>0.1&|y|<=0.1 : cost increase same unserved energy (very bad)

In [ ]:
quadrant_df

In [ ]:

# Option 1: Scatter with marginal distributions
# import plotly.figure_factory as ff
# fig = ff.create_scatterplotmatrix(
#     quadrant_df[[g_MODEL_VARIABLE, 'OV_uc_diff_%', 'E_ENS_MWh_diff_%']],
#     diag='histogram',
#     index=g_MODEL_VARIABLE,
#     height=800, width=800
# )
# fig.show()

# # Option 2: 2D hexbin-style visualization
fig = px.density_contour(
    quadrant_df, 
    x='OV_uc_diff_%', 
    y='E_ENS_MWh_diff_%',
    color=g_MODEL_VARIABLE,
    marginal_x='histogram', 
    marginal_y='histogram'
)
fig.show()

# Option 3: Correlation analysis by model type
print(quadrant_df.groupby(g_MODEL_VARIABLE)[['OV_uc_diff_%', 'E_ENS_MWh_diff_%']].corr())

# Option 4: Violin plots for distributions
# fig = px.violin(
#     differences.melt(id_vars=[g_MODEL_VARIABLE], 
#                      value_vars=['OV_uc_diff_%', 'E_ENS_MWh_diff_%']),
#     x=g_MODEL_VARIABLE,
#     y='value',
#     facet_col='variable',
#     color=g_MODEL_VARIABLE,
# )
# fig.show()